In [1]:
%%html
<style>
    h1 {color:purple}
    h2 {color:purple}
    h3 {color:#0099ff}
    hr {    
        border: 0;
        height: 3px;
        background: #333;
        background-image: linear-gradient(to right, limegreen, deepskyblue, limegreen);
    }
</style>

# Creating Agents with the OpenAI Agents SDK
---
# Streaming Text and Events
## Three Top-Level `StreamEvent.type` Values
* From `Runner.run_streamed(...).stream_events()`
* https://openai.github.io/openai-agents-python/ref/stream_events/ 

**`raw_response_event`**
* Low-level streaming events from Responses API
    * Many `event.data.type` values
* Provides token/text deltas, such as `ResponseTextDeltaEvent` / `response.output_text.delta`
    * "Someone is typing a response" effect

**`run_item_stream_event`**
* Higher-level Agents SDK events
* Occur when agent completes a run item
    * E.g., message, tool call, tool output, handoff, reasoning, MCP event

**`agent_updated_stream_event`**
* Occurs when the current agent changes, typically for a handoff
---

### Agents SDK `run_item_stream_event` Event Names

```text
message_output_created
handoff_requested
handoff_occured  
tool_called
tool_search_called
tool_search_output_created
tool_output
reasoning_item_created
mcp_approval_requested
mcp_approval_response
mcp_list_tools
```
* **`handoff_occured` is intentionally misspelled** 
    * For backward compatibility
    * https://openai.github.io/openai-agents-python/streaming/ 

## Python Tutor with Streamed Responses

In [2]:
import asyncio

from agents import Agent, Runner, trace
from openai.types.responses import ResponseTextDeltaEvent
from IPython.display import HTML, Markdown, display

# create an Agent
tutor = Agent(
    name='Python Tutor',
    model='gpt-6.1-terra', # gpt-5.6-luna is the current SDK default
    instructions="""You are a Python tutor for novice programmers. 
        Provide concise, simple answers to Python questions."""
)

In [ ]:
print('I am a Python tutor. How can I help you?') 
turn_count = 1 # turn counter for user-input prompts
previous_response_id = None # used to chain responses for context

# FOR DEMO PURPOSES ONLY
# seconds to pause after each text chunk so streaming is easy to see in
# class; set to 0 to show the response at the model's full speed
STREAM_DELAY = 0.05

# run until the user input is 'exit'
while ((prompt := input(f'\n Input [{turn_count}]: ')) != 'exit'):
    with trace(f'02-03-streaming-response-[{turn_count}]'):
        # launch agent asynchronously and stream its response
        result = Runner.run_streamed(tutor, prompt,
            # continue from the previous turn's response (None on the first turn)
            previous_response_id=previous_response_id,
            # also chain model calls within this run, even on the first turn;
            # good practice; more for examples with tool calls and hand-offs
            auto_previous_response_id=True
        )

        # blank line in conversation after user input
        display(HTML('<div style="height: 1em;"/>'))
        markdown_text = ''

        # handle enables updating output below this code cell
        handle = display(Markdown(markdown_text), display_id=True)
        
        # print text deltas as they arrive, similar to ChatGPT streaming
        async for event in result.stream_events():
            # if the event is part of the text response, display it; 
            # provides effect of someone typing a response to you
            if (event.type == 'raw_response_event' and 
                isinstance(event.data, ResponseTextDeltaEvent)): 
                markdown_text += event.data.delta
                handle.update(Markdown(markdown_text))
                # FOR DEMO PURPOSES ONLY
                await asyncio.sleep(STREAM_DELAY) # slow the display for demos

        turn_count += 1 
        display(HTML('<hr>'))

        # save last response ID for context during the next turn
        previous_response_id = result.last_response_id

print('User terminated app.')

I am a Python tutor. How can I help you?



 Input [1]:  Given a list containing 1-10, what's the easiest way to total them? Also, show me other ways and give me the pros and cons of each.


The easiest way is Python’s built-in `sum()`:

```python
numbers = list(range(1, 11))  # 1 through 10
total = sum(numbers)

print(total)  # 55
```

**Pros:** Short, clear, and works with any list of numbers.  
**Cons:** Less flexible if you need to process each number differently.

### Other ways

**1. A `for` loop**

```python
total = 0
for number in numbers:
    total += number
```

**Pros:** Easy to customize, such as skipping certain numbers.  
**Cons:** More code than `sum()`.

**2. `reduce()`**

```python
from functools import reduce

total = reduce(lambda total, number: total + number, numbers, 0)
```

**Pros:** Can combine items using more complex operations.  
**Cons:** Harder to read; unnecessary for simple addition.

**3. A formula**

```python
n = 10
total = n * (n + 1) // 2
```

**Pros:** Fast and doesn’t require a list.  
**Cons:** Only works for consecutive integers from 1 through `n`.

For a list you already have, **use `sum(numbers)`**.

---

## Sample Input for This Demo
* Given a list containing 1-10, what's the easiest way to total them? Also, show me other ways and give me the pros and cons of each.
---

## Useful Inspection Pattern

```python
async for event in result.stream_events():
    print(event.type)

    if event.type == 'run_item_stream_event':
        print('  name:', event.name)
        print('  item type:', event.item.type)

    elif event.type == 'agent_updated_stream_event':
        print('  new agent:', event.new_agent.name)

    elif event.type == 'raw_response_event':
        print('  raw event:', event.data.type)
```


---
&copy; 1992-2026 by Deitel & Associates, Inc. and Pearson Education, Inc. All Rights Reserved. 